In [ ]:
import re
import pandas as pd
import torch
import numpy as np
from tqdm import tqdm
from torch.nn.functional import pad
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import matplotlib.colors as mcolors

In [ ]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from transformers import LlamaForCausalLM, LlamaTokenizerFast

import argparse
import pickle
import os
import json
import pandas as pd

#model_id = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"

model_id = "gpt2-large"
device='cuda:1'
if "Llama" in model_id:
    #model = LlamaForCausalLM.from_pretrained(model_id, device_map="auto")
    tokenizer = LlamaTokenizerFast.from_pretrained(model_id)
    start_of_sentence='<|begin_of_text|>'
if "gpt2" in model_id:
    model = GPT2LMHeadModel.from_pretrained(model_id).to(device)
    tokenizer = GPT2TokenizerFast.from_pretrained(model_id)
    start_of_sentence =" "

In [ ]:
import os
os.chdir("/u/sebono/conversational_dominance/information_exchange_labelling")
os.getcwd()

In [ ]:
from perplexity_labelling import compute_p1, compute_p2, compute_p3
from utils import swap_speaker_tokens_with_metadata, get_shuffled_dialog_data, reorder_tokens

In [ ]:
import numpy as np
import re
import pandas as pd

In [ ]:
tri_pilot_df = pd.read_csv("/u/sebono/conversational_dominance/data/processed/triadic-pilot/conversations.csv")
tri_pilot_df

In [ ]:
file_name = "p08_s03"
dialog = tri_pilot_df[tri_pilot_df['file_name'] == file_name]['file_content'][0]

In [ ]:
pattern = r'<(?:SPK[0-9]|MOD)>'
dialog_lines = re.sub(r"[\[\(].*?[\]\)]", "", dialog).replace("<", "\n<").split("\n")[1:]
matches = [f"{start_of_sentence}{match}" for match in re.findall(pattern, f"{start_of_sentence}".join(dialog_lines))]
token_list = [tokenizer(token, return_tensors="pt").input_ids[0] for token in dialog_lines]
encodings = tokenizer(f"{start_of_sentence}".join(dialog_lines), return_tensors="pt")
assert np.cumsum([len(token) for token in token_list])[-1] == encodings.input_ids[0].shape

In [ ]:
assert len(matches) == len(dialog_lines)

In [ ]:
from utils import correlation_heatmap

In [ ]:
all_data = pd.DataFrame({})

In [ ]:
all_data = pd.DataFrame({})
for p_name in ['p1','p2','p3']:
    file_name_sample = f'/u/sebono/conversational_dominance/notebooks/information_exchange_labelling/dataset_perplexity_results/triadic-pilot_{p_name}_{model_id}/dominance_scores_{file_name}.pkl'
    with open(file_name_sample, 'rb') as f:
        all_data[f"{p_name}"] = pickle.load(f)[file_name]

In [ ]:
from utils import correlation_heatmap, correlation_heatmap, compute_dominance_per_spk, rolling_kde_heatmap_with_turns, remove_match_prefix_ppl, compute_graph_perplexity, display_turns_colored_by_kde

In [ ]:
all_data

In [ ]:
# BEFORE
col_1 = ["p1","p2","p3"]
corr, fig_corr, p, fig_p, fig_r = correlation_heatmap(col_1,col_1,all_data)

In [ ]:
fig_corr

In [ ]:
from utils import remove_match_prefix_ppl, display_colored_sentences, extract_tokens_and_ppls_by_turn_indices, correlation_heatmap

In [ ]:
filtered_tokens_p1, filtered_ppl_p1, filtered_encodings_p1, filtered_matches_p1 = remove_match_prefix_ppl(token_list, all_data['p1'], matches, tokenizer, min_len=1)
assert np.cumsum([len(token) for token in filtered_tokens_p1])[-1], len(filtered_tokens_p1)

In [ ]:
filtered_tokens_p2, filtered_ppl_p2, filtered_encodings_p2, filtered_matches_p2 = remove_match_prefix_ppl(token_list, all_data['p2'], matches, tokenizer, min_len=1)
assert np.cumsum([len(token) for token in filtered_tokens_p2])[-1], len(filtered_tokens_p2)

In [ ]:
filtered_tokens_p3, filtered_ppl_p3, filtered_encodings_p3, filtered_matches_p3 = remove_match_prefix_ppl(token_list, all_data['p3'], matches, tokenizer, min_len=1)
assert np.cumsum([len(token) for token in filtered_tokens_p3])[-1], len(filtered_tokens_p3)

In [ ]:
assert len(filtered_tokens_p1) == len(filtered_tokens_p2)
assert len(filtered_tokens_p1) == len(filtered_tokens_p3)

assert filtered_encodings_p1 == filtered_encodings_p2
assert filtered_encodings_p1 == filtered_encodings_p3
filtered_encodings = filtered_encodings_p1

In [ ]:
# BEFORE
all_data_filtered = pd.DataFrame({"p1":filtered_ppl_p1, "p2":filtered_ppl_p2, "p3":filtered_ppl_p3})
col_1 = ["p1","p2","p3"]
corr, fig_corr, p, fig_p, fig_r = correlation_heatmap(col_1,col_1,all_data_filtered)

In [ ]:
fig_corr

In [ ]:
# Get turn indices for SPK0 (kid)
start_turn=0
end_turn=-1
idx_spk0 = start_turn + np.where(np.asarray(filtered_matches_p2)[start_turn:end_turn] == f"{start_of_sentence}<SPK0>")[0]

# Get filtered tokens and PPLs
filtered_tokens_p2_spk0, filtered_ppl_p2_spk0 = extract_tokens_and_ppls_by_turn_indices(
    idx_spk0, filtered_tokens_p2, filtered_ppl_p2
)

In [ ]:
# Get turn indices for SPK0 (kid)
start_turn=0
end_turn=-1
idx_spk0 = start_turn + np.where(np.asarray(filtered_matches_p3)[start_turn:end_turn] == f"{start_of_sentence}<SPK0>")[0]

# Get filtered tokens and PPLs
filtered_tokens_p3_spk0, filtered_ppl_p3_spk0 = extract_tokens_and_ppls_by_turn_indices(
    idx_spk0, filtered_tokens_p3, filtered_ppl_p3
)

In [ ]:
display_colored_sentences(filtered_tokens_p3_spk0, filtered_ppl_p3_spk0, tokenizer)

In [ ]:
display_colored_sentences(filtered_tokens_p2_spk0, filtered_ppl_p2_spk0, tokenizer)

### Annotations

In [ ]:
import os
os.chdir("/u/sebono/conversational_dominance/information_exchange_labelling")
print(os.getcwd())

from utils import (
    # VISUALIZATION
    rolling_kde_heatmap_with_turns,
    display_turns_colored_by_kde,
    compute_graph_perplexity,
    correlation_heatmap,
    # HELPERS
    compute_dominance_per_spk,
    compute_significance,
    expand_multiple_ppl_by_token,
    remove_match_prefix_ppl,
    assign_words_to_bins
)

In [ ]:
annotation_file = f"/u/sebono/conversational_dominance/data/processed/triadic-pilot/annotation_files/childs-joint-engagement-with-parent/aggregated_annotation_files.csv"

In [ ]:
annotations = pd.read_csv(annotation_file)
annotations_dataset = annotations[annotations['uniq_vid_clip'].str.startswith(file_name)]
annotations_dataset.head()

In [ ]:
conversation_dataset_file = f'/u/sebono/conversational_dominance/data/external/triadic-pilot/{file_name}_REV.csv'
conversation_dataset = pd.read_csv(conversation_dataset_file)
conversation_dataset

In [ ]:
assert len(matches) == len(conversation_dataset['content'])

In [ ]:
conversation_dataset["tokens"] = [token.detach().numpy() for token in filtered_tokens_p1]
conversation_dataset[conversation_dataset['speaker']=='Child']

In [ ]:
import pandas as pd

#df = conversation_dataset[conversation_dataset['speaker']=='Child'].copy()
df = conversation_dataset.copy()
df["start"] = pd.to_timedelta(df["timestamp"]).dt.total_seconds()
df["stop"] = df["start"].shift(-1)
df.loc[df.index[-1], "stop"] = df["start"].iloc[-1] + 1
df = df[["start", "stop", "speaker", "content", "tokens"]]

In [ ]:
df

In [ ]:
import numpy as np
import pandas as pd
import torch
from collections import defaultdict

def assign_words_to_bins(df, bin_size=5.0):
    """
    Assigns tokens to time bins based on uniform spread over the utterance duration.
    Includes utterances with 0 duration. Ensures all bins from time 0 to max stop are represented.

    Args:
        df (pd.DataFrame): DataFrame with columns ['start_time', 'end_time', 'tokens']
        bin_size (float): Size of each time bin in seconds

    Returns:
        pd.DataFrame: each row is a bin with its start/end time, decoded words, token indices, and token count
    """
    bin_tok = defaultdict(list)
    bin_ppl = defaultdict(list)
    global_token_idx = 0

    for _, row in df.iterrows():
        start = row["start"]
        stop = row["stop"]
        tokens = row["tokens"]

        if pd.isna(start) or pd.isna(stop) or tokens is None or len(tokens) == 0:
            continue

        duration = stop - start
        if duration < 0:
            continue

        # Handle 0-duration: treat as a very short utterance
        if duration == 0:
            duration = 1e-6

        # Token timing interpolation
        tok_times = np.linspace(start, stop, len(tokens) + 1)

        for i, token in enumerate(tokens):
            tok_start = tok_times[i]
            tok_end = tok_times[i + 1]

            bin_start_idx = int(np.floor(tok_start / bin_size))
            bin_end_idx = int(np.floor((tok_end - 1e-6) / bin_size))  # avoid border collision

            for b in range(bin_start_idx, bin_end_idx + 1):
                bin_tok[b].append(token)
                bin_ppl[b].append(global_token_idx + i)

        global_token_idx += len(tokens)

    # Build output DataFrame
    max_bin = int(np.ceil(df["stop"].max() / bin_size))
    all_bins = list(range(max_bin + 1))

    bins_df = pd.DataFrame([
        {
            "time_bin": b,
            "start_time": b * bin_size,
            "end_time": (b + 1) * bin_size,
            "words": tokenizer.decode(torch.tensor(bin_tok[b]), skip_special_tokens=True),
            "ppl": bin_ppl[b],
            "tokens": torch.tensor(bin_tok[b]),
            "n_tok": len(bin_tok[b])
        }
        for b in all_bins
    ])

    return bins_df

In [ ]:
bins_df = assign_words_to_bins(df, bin_size=5.0)

In [ ]:
bins_df

In [ ]:
combined_df = pd.concat([bins_df.reset_index(drop=True), annotations_dataset.iloc[:bins_df.shape[0]].reset_index(drop=True)], axis=1)
combined_df.iloc[0:]

In [ ]:
cols = ['Rating_x', 'Rating_y','start_time','end_time','words']
ppl_dict = {'p1': all_data_filtered["p1"], 'p2': all_data_filtered["p2"], 'p3': all_data_filtered["p3"]}
all_data_filtered_expanded = expand_multiple_ppl_by_token(combined_df, ppl_dict, annotation_cols=cols)

In [ ]:
from sklearn.decomposition import PCA
all_data_filtered_expanded.ffill(inplace=True)
X = np.column_stack((all_data_filtered_expanded["p2"],all_data_filtered_expanded["p3"]))

In [ ]:
#KMeans (assumes spherical clusters):
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=0)
labels = kmeans.fit_predict(X)

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap="tab10", s=40)
plt.title("Clusters of Turns in P2/P3 PCA Space")
plt.xlabel("PC2")
plt.ylabel("PC3")
plt.colorbar(label="Cluster Label")
plt.grid(True)
plt.show()

In [ ]:
all_data_filtered_expanded[:50]

In [ ]:
for cluster_id in np.unique(labels):
    print(f"\n--- Cluster {cluster_id} ---")
    indices = np.where(labels == cluster_id)[0][:10]
    print(all_data_filtered_expanded.iloc[indices]['words'].to_numpy())

In [ ]:
# AFTER
col_1 = ["p2","p3","p1"]
col_2 = ['Rating_x','Rating_y']
corr, fig_corr, p, fig_p, fig_r = correlation_heatmap(col_1,col_2,all_data_filtered_expanded)

In [ ]:
fig_corr

In [ ]:
def average_ppl_per_turn(df, ppl_dict, idx_col='ppl', annotation_cols=None):
    """
    For each row in df, average values from ppl_dict at indices listed in df[idx_col].

    Args:
        df (pd.DataFrame): DataFrame with a column of token indices (e.g., 'ppl').
        ppl_dict (dict): Dictionary with name -> list of PPL values.
        idx_col (str): Column in df with list of indices per turn.
        annotation_cols (List[str]): Optional columns to carry forward.

    Returns:
        pd.DataFrame: Averaged PPL values per turn, optionally with annotations.
    """
    import numpy as np
    import pandas as pd

    records = []
    for _, row in df.iterrows():
        token_ids = row[idx_col]
        if not isinstance(token_ids, list) or len(token_ids) == 0:
            continue

        record = {}
        for name, ppl_values in ppl_dict.items():
            selected = [ppl_values[i] for i in token_ids if i < len(ppl_values)]
            record[f'{name}_avg'] = np.mean(selected) if selected else None
            record['tokens'] = token_ids
            record['tok_len'] = len(token_ids)

        if annotation_cols:
            for col in annotation_cols:
                record[col] = row.get(col, None)

        records.append(record)

    return pd.DataFrame(records)

In [ ]:
all_data_filtered

In [ ]:
cols = ['Rating_x', 'Rating_y','words','start_time','end_time', 'tokens']
ppl_dict = {'p1': all_data_filtered["p1"], 'p2': all_data_filtered["p2"], 'p3': all_data_filtered["p3"]}
all_data_filtered_expanded = average_ppl_per_turn(combined_df, ppl_dict, annotation_cols=cols)

In [ ]:
all_data_filtered_expanded

In [ ]:
sum(all_data_filtered_expanded['Rating_x']==1), sum(all_data_filtered_expanded['Rating_x']==0), sum(all_data_filtered_expanded['Rating_x']==2), sum(all_data_filtered_expanded['Rating_x']==-2), sum(all_data_filtered_expanded['Rating_x']==-1)

In [ ]:
from utils import correlation_heatmap

In [ ]:
# AFTER
col_1 = ["p1_avg","p2_avg","p3_avg"]
col_2 = ['Rating_x','Rating_y']
corr, fig_corr, p, fig_p, fig_r = correlation_heatmap(col_1,col_2,all_data_filtered_expanded)

In [ ]:
fig_r

In [ ]:
fig_corr

In [ ]:
fig_p

In [ ]:
# p2 --> high ppl, less predictable utterace --> more topical dominance
# A: hi my name is Serena.| B: I don't care, get back to work! --> very unliklely utterance, really high ppl, B: is dominant
# p3 --> high ppl, less likely to be interrupted --> more interactional dominance
# A: hi my name is Serena.| B: I don't care, [:A] --> very unliklely interruption, really high ppl, B: is dominant
# follows that dominance should be directly correlated w. perplexity.
# engagement tho... it depends. If the roles are child/parent, then an imbalanced conversation is to be expected.

# interruptions are inversely correlated to engagement?
# topical dominance is directly correlated to engagement?
# Is this general, likely not!

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Drop NaNs
all_data_filtered_expanded.dropna(inplace=True)

# Features
X = all_data_filtered_expanded[['p2_avg', 'p3_avg']]

# Binarize target
y = (all_data_filtered_expanded['Rating_x'] >= 1).astype(int)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train logistic regression
model = LogisticRegression(class_weight='balanced')
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


In [ ]:
import pandas as pd

feature_names = ['p2_avg', 'p3_avg']
coeffs = model.coef_[0]

for name, val in zip(feature_names, coeffs):
    print(f"{name}: {val:.4f}")


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

y_proba = model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_proba):.2f}")
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
import matplotlib.pyplot as plt

# Use the same X and y from before
lda = LDA(n_components=1)  # Only 1 component for binary classification
X_lda = lda.fit_transform(X, y)

# Plot
plt.figure(figsize=(8, 4))
plt.scatter(X_lda[y==0], [0]*sum(y==0), c='red', label='Class 0 (Rating < 1)')
plt.scatter(X_lda[y==1], [1]*sum(y==1), c='green', label='Class 1 (Rating ≥ 1)')
plt.title("LDA Projection")
plt.yticks([0, 1], ['Class 0', 'Class 1'])
plt.xlabel("LDA Component 1")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
X

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X[['p2_avg','p3_avg']])

plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='coolwarm', edgecolor='k')
plt.title("PCA Projection")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True)
plt.colorbar(label='Class (0/1)')
plt.show()


In [ ]:
all_data_filtered_expanded

In [ ]:
repeated_p2_list = sum([[p2] * l for p2, l in zip(all_data_filtered_expanded['p2_avg'], all_data_filtered_expanded['tok_len'])], [])

In [ ]:
repeated_p3_list = sum([[p3] * l for p3, l in zip(all_data_filtered_expanded['p3_avg'], all_data_filtered_expanded['tok_len'])], [])

In [ ]:
assert all_data_filtered_expanded["tok_len"].sum() == len(repeated_p2_list)

In [ ]:
display_colored_sentences(list(all_data_filtered_expanded['tokens']), repeated_p2_list, tokenizer)

In [ ]:
display_colored_sentences(list(all_data_filtered_expanded['tokens']), repeated_p3_list, tokenizer)

### Clusters

In [ ]:
all_data_filtered_expanded["p2_avg"]

In [ ]:
from sklearn.decomposition import PCA
all_data_filtered.ffill(inplace=True)
X = np.column_stack((all_data_filtered_expanded["p2_avg"],all_data_filtered_expanded["p3_avg"]))

In [ ]:
all_data_filtered_expanded

In [ ]:
#KMeans (assumes spherical clusters):
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=0)
labels = kmeans.fit_predict(X)

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap="tab10", s=40)
plt.title("Clusters of Turns in P2/P3 PCA Space")
plt.xlabel("PC2")
plt.ylabel("PC3")
plt.colorbar(label="Cluster Label")
plt.grid(True)
plt.show()

In [ ]:
for cluster_id in np.unique(labels):
    print(f"\n--- Cluster {cluster_id} ---")
    indices = np.where(labels == cluster_id)[0][:10]
    print(all_data_filtered_expanded.iloc[indices]['words'].to_numpy())


### Finding Signal

In [ ]:
from utils import compute_dominance_per_spk, compute_significance

In [ ]:
baselined_ppls_p2_spk = compute_dominance_per_spk(all_data_filtered["p2"], filtered_tokens_p1, matches, tokenizer)
baselined_ppls_p3_spk = compute_dominance_per_spk(all_data_filtered["p3"], filtered_tokens_p1, matches, tokenizer)

In [ ]:
all_data_filtered_expanded

In [ ]:
from scipy.stats import spearmanr
import pandas as pd

cols = ['Rating_x','Rating_y']
# Perform correlation
stat, pval = spearmanr(all_data_filtered_expanded['p3_avg'], all_data_filtered_expanded[cols], nan_policy='omit')

# Extract just the relevant row (first row after the scalar)
correlation_vector = stat[0, 1:]
pval_vector = pval[0, 1:]

# Wrap into a DataFrame
corr_df = pd.DataFrame({
    "feature": cols,
    "spearman_r": correlation_vector,
    "p_value": pval_vector
})

# Sort by absolute correlation
corr_df["abs_r"] = corr_df["spearman_r"].abs()
corr_df = corr_df.sort_values("abs_r", ascending=False)

corr_df[["feature", "spearman_r", "p_value"]]


In [ ]:
X = all_data_filtered_expanded[cols].fillna(0)
y = all_data_filtered_expanded['p3_avg']

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X, y)

rf_importances = pd.DataFrame({
    "feature": cols,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

print(rf_importances)

In [ ]:
from sklearn.inspection import partial_dependence, PartialDependenceDisplay

PartialDependenceDisplay.from_estimator(rf, X, ['Rating_y'])
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split
# Features and target
all_data_filtered_expanded.dropna(inplace=True)
X = all_data_filtered_expanded[cols]
y = all_data_filtered_expanded["p3_avg"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
!pip install shap

In [ ]:
import shap
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor().fit(X_train, y_train)
explainer = shap.Explainer(rf, X_train)
shap_values = explainer(X_test)
shap.plots.beeswarm(shap_values)